# RetinaXAI - Diabetic Retinopathy Lesion Segmentation and Explainability Benchmark

This notebook is designed to run end to end in a single Google Colab session (T4 GPU)
and is split into two complementary tracks that share the same IDRiD lesion masks and
the same IoU / Dice evaluation protocol.

**Track A - Supervised lesion segmentation (primary metric engine).**
A U-Net++ encoder-decoder with a pretrained EfficientNet encoder is trained directly
on the IDRiD pixel-level lesion masks (microaneurysms, haemorrhages, hard exudates,
soft exudates). This is the architecture that actually pushes IoU and Dice up, because
it is optimised on pixel-level ground truth with a loss designed for tiny, heavily
imbalanced lesions.

**Track B - Explainability benchmark (preserved project identity).**
A classifier trained on APTOS-2019 severity labels is probed with Grad-CAM,
Grad-CAM++, Layer-CAM, HiRes-CAM and Eigen-CAM. These weakly-supervised saliency maps
are scored against the exact same IDRiD masks so we can quantify how far explainability
heatmaps sit from a fully supervised segmenter.

## Why the previous Grad-CAM-only version scored so low

The earlier pipeline read heatmaps from a classifier and compared them to pixel-level
lesion masks. Measured IoU was 0.005 - 0.10 and Dice 0.01 - 0.18, with Eigen-CAM the
worst. This is not a tuning bug - it is the documented ceiling of weakly-supervised
localisation:

- Class Activation Maps highlight the regions a classifier finds *discriminative*, not
  the precise extent of every lesion. For an image graded "severe", a small cluster of
  microaneurysms is enough to drive the decision, so the CAM lights up a coarse blob
  and ignores the other lesions.
- Eigen-CAM is especially unsuited here: it returns the first principal component of the
  target-layer activations and does **not** use the class/target at all, so its map is
  not lesion-specific. Poor Eigen-CAM overlap is expected, not fixable by thresholding.
- Microaneurysms span roughly 15-60 micrometres (a handful of pixels). Even the 14x14
  feature map we upsampled from is far coarser than the lesions we are scoring against.

The rational fix, confirmed by the IDRiD literature, is to train a dedicated segmentation
network. We keep the CAM benchmark as an honest comparison rather than as the main
localiser.


## 0. Environment setup

Mount Google Drive (all data, checkpoints and outputs live there so nothing is lost when
the Colab VM disconnects) and install the libraries that are not preinstalled on Colab.


In [ ]:
# Mount Drive. Wrapped in a guard so the notebook also runs on a local machine.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    # Not on Colab (e.g. local run): we simply skip mounting.
    IN_COLAB = False
    print('Not running on Colab - Drive mount skipped.')


In [ ]:
# segmentation-models-pytorch  -> U-Net++ / DeepLabV3+ decoders with pretrained encoders
# albumentations                -> fast image+mask synchronised augmentation
# grad-cam                      -> Grad-CAM family for the explainability track
# We pin nothing aggressively; -q keeps the log short.
!pip install -q segmentation-models-pytorch==0.3.3 albumentations==1.4.0 grad-cam==1.5.0


In [ ]:
import os
import gc
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

warnings.filterwarnings('ignore')

# Device selection. On Colab this is the T4 GPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))
    print('VRAM  : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))


In [ ]:
def set_seed(seed=42):
    '''Seed every RNG we rely on so a resumed run is reproducible.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Deterministic cuDNN trades a little speed for reproducibility; acceptable here.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


## 1. Configuration

Everything you might want to tune lives in one place. Paths follow the folder layout the
project already used on Drive; adjust `BASE` if yours differs.

The IDRiD "Segmentation" subset ships 81 fundus images with pixel-level masks, split by
the organisers into 54 training and 27 testing images. We keep that official split and
carve a small validation slice out of the training images for early stopping.


In [ ]:
class CFG:
    # ---- Paths (edit BASE to match your Drive) ----------------------------
    BASE = '/content/drive/MyDrive/retinaxai' if IN_COLAB else './retinaxai_data'
    IDRID_BASE   = f'{BASE}/data/idrid'
    IDRID_IMG_DIR = f'{IDRID_BASE}/images'            # IDRiD_XX.jpg fundus images
    IDRID_MASK_DIR = f'{IDRID_BASE}/masks'            # masks/<LesionName>/IDRiD_XX_<SUF>.tif

    # APTOS (only needed for Track B - the explainability classifier)
    APTOS_CSV = f'{BASE}/data/aptos2019/subset_large.csv'
    APTOS_IMG_DIR = f'{BASE}/data/aptos2019/subset_images_large'

    OUTPUT_DIR = f'{BASE}/outputs'
    CKPT_DIR   = f'{BASE}/checkpoints'                # checkpoints survive disconnects here

    # ---- Lesion channels (multi-label: a pixel can belong to several) ------
    # Folder name -> IDRiD filename suffix. Order defines the output channels.
    LESIONS = {
        'Microaneurysms': 'MA',
        'Haemorrhages':   'HE',
        'HardExudates':   'EX',
        'SoftExudates':   'SE',
    }

    # ---- Image / model ----------------------------------------------------
    IMG_SIZE   = 512                 # 512 balances lesion detail vs T4 memory
    ENCODER    = 'efficientnet-b4'   # pretrained encoder for the U-Net++ decoder
    ENCODER_WEIGHTS = 'imagenet'
    ARCH       = 'unetplusplus'      # 'unetplusplus' or 'deeplabv3plus'
    NUM_LESIONS = len(LESIONS)

    # ---- Training ---------------------------------------------------------
    BATCH_SIZE = 4                   # B4 @ 512 with AMP fits a 15 GB T4 at bs=4
    GRAD_ACCUM = 2                   # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
    NUM_EPOCHS = 80
    LR         = 3e-4                # OneCycle max LR for the AdamW optimiser
    WEIGHT_DECAY = 1e-4
    PATIENCE   = 15                  # early-stopping patience on validation Dice
    NUM_WORKERS = 2

    # ---- Loss weighting ---------------------------------------------------
    # Focal-Tversky pushes recall on tiny lesions; Dice stabilises the overlap.
    TVERSKY_ALPHA = 0.7              # weight on false negatives (>0.5 favours recall)
    TVERSKY_BETA  = 0.3             # weight on false positives
    TVERSKY_GAMMA = 0.75            # focal exponent (<1 focuses on hard, rare pixels)
    LOSS_DICE_W    = 0.5
    LOSS_TVERSKY_W = 0.5

    # ---- Evaluation -------------------------------------------------------
    THRESHOLD = 0.5                  # default probability -> mask threshold

    # ---- Preprocessing toggles -------------------------------------------
    USE_CLAHE = True                 # CLAHE on the green channel boosts lesion contrast

    SEED = 42

cfg = CFG()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
os.makedirs(cfg.CKPT_DIR, exist_ok=True)
print('Outputs     ->', cfg.OUTPUT_DIR)
print('Checkpoints ->', cfg.CKPT_DIR)


## 2. Fundus preprocessing

Two evidence-based steps from the retinal-imaging literature:

1. **Retina crop** - fundus photos carry a wide black border. Cropping to the circular
   field of view removes uninformative background so the lesions occupy more of the frame.
2. **CLAHE on the green channel** - the green channel has the highest contrast between
   retinal lesions and background, and Contrast Limited Adaptive Histogram Equalisation
   locally amplifies the faint microaneurysms and exudates without blowing out bright
   regions. We apply CLAHE to green, keep red and blue, and recombine.

Both are applied identically to images and (for the crop) to masks so pixels stay aligned.


In [ ]:
def crop_retina_bbox(image_np):
    '''Return the bounding box (r0, r1, c0, c1) of the circular fundus region.

    We threshold a blurred grayscale version to separate the illuminated retina from
    the black border, then take the tightest box around it. Returning the box (instead
    of the cropped image) lets us apply the exact same crop to every lesion mask.
    '''
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (7, 7), 0)
    # A low absolute threshold is enough because the border is near-black.
    mask = gray > 10
    if mask.sum() == 0:
        # Degenerate image: fall back to the full frame.
        return 0, image_np.shape[0], 0, image_np.shape[1]
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    return rows[0], rows[-1] + 1, cols[0], cols[-1] + 1


def apply_clahe_green(image_np, clip_limit=2.0, grid=(8, 8)):
    '''CLAHE on the green channel; red and blue are passed through unchanged.'''
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid)
    out = image_np.copy()
    out[:, :, 1] = clahe.apply(out[:, :, 1])
    return out


def load_fundus(image_path, use_clahe=True):
    '''Load an RGB fundus image, crop to the retina, optionally CLAHE it, and also
    return the crop box so the matching masks can be cropped the same way.'''
    image = np.array(Image.open(image_path).convert('RGB'))
    r0, r1, c0, c1 = crop_retina_bbox(image)
    image = image[r0:r1, c0:c1]
    if use_clahe:
        image = apply_clahe_green(image)
    return image, (r0, r1, c0, c1)


## 3. IDRiD segmentation dataset

Robustness details that matter for a small (81-image) medical set on an unreliable VM:

- **Missing masks are normal.** Not every image contains every lesion type, so a missing
  `.tif` is silently treated as an all-zero channel rather than an error.
- **Multi-label, not multi-class.** Lesions overlap, so each of the four lesion types is
  its own binary channel (sigmoid output) instead of a single softmax label per pixel.
- **Synchronised augmentation.** Albumentations transforms the image and all four mask
  channels together, so flips/rotations never desynchronise a lesion from its label.


In [ ]:
class IDRiDSegmentationDataset(Dataset):
    '''Loads a fundus image plus its four lesion masks as a multi-label target.

    Returns:
        image : float tensor  [3, H, W]  (ImageNet-normalised)
        mask  : float tensor  [4, H, W]  (0/1 per lesion channel)
        image_id : str
    '''
    def __init__(self, image_ids, cfg, transform, use_clahe=True):
        self.ids = image_ids
        self.cfg = cfg
        self.transform = transform
        self.use_clahe = use_clahe

    def __len__(self):
        return len(self.ids)

    def _load_mask_channel(self, image_id, lesion_name, suffix, crop_box, out_hw):
        '''Load one lesion mask, crop it to match the image, or return zeros.'''
        path = f'{self.cfg.IDRID_MASK_DIR}/{lesion_name}/{image_id}_{suffix}.tif'
        if not os.path.exists(path):
            return np.zeros(out_hw, dtype=np.uint8)
        m = np.array(Image.open(path).convert('L'))
        r0, r1, c0, c1 = crop_box
        m = m[r0:r1, c0:c1]
        return (m > 0).astype(np.uint8)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        img_path = f'{self.cfg.IDRID_IMG_DIR}/{image_id}.jpg'
        image, crop_box = load_fundus(img_path, self.use_clahe)
        hw = image.shape[:2]

        # Stack the four lesion channels into an (H, W, 4) mask for Albumentations.
        masks = [
            self._load_mask_channel(image_id, name, suf, crop_box, hw)
            for name, suf in self.cfg.LESIONS.items()
        ]
        mask = np.stack(masks, axis=-1).astype('float32')

        augmented = self.transform(image=image, mask=mask)
        image_t = augmented['image']                      # [3, H, W] float
        mask_t = augmented['mask'].permute(2, 0, 1)       # [4, H, W] float
        return image_t, mask_t, image_id


In [ ]:
# ImageNet statistics because the encoder is ImageNet-pretrained.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Training augmentation: geometric flips/rotations are safe for fundus images (there is
# no canonical orientation), and mild photometric jitter simulates camera/illumination
# variation. Elastic/grid distortion helps the model generalise from only ~46 images.
train_tf = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20,
                       border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# Validation / test: only resize + normalise, so metrics reflect real performance.
eval_tf = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


In [ ]:
def build_idrid_splits(cfg, val_frac=0.15):
    '''Return (train_ids, val_ids, test_ids).

    IDRiD's segmentation subset is officially split into training images (IDRiD_01..54)
    and testing images (IDRiD_55..81). We honour that when filenames follow the pattern,
    and otherwise fall back to a deterministic ratio split so the notebook still runs on
    a differently organised copy of the data.
    '''
    all_ids = sorted(
        f[:-4] for f in os.listdir(cfg.IDRID_IMG_DIR) if f.lower().endswith('.jpg')
    )
    assert len(all_ids) > 0, f'No .jpg images found in {cfg.IDRID_IMG_DIR}'

    def idrid_number(image_id):
        # 'IDRiD_07' -> 7 ; returns None if the pattern does not match.
        try:
            return int(image_id.split('_')[-1])
        except ValueError:
            return None

    numbers = [idrid_number(i) for i in all_ids]
    if all(n is not None for n in numbers):
        train_pool = [i for i, n in zip(all_ids, numbers) if n <= 54]
        test_ids   = [i for i, n in zip(all_ids, numbers) if n >= 55]
    else:
        # Fallback: deterministic 80/20 split.
        rng = np.random.RandomState(cfg.SEED)
        perm = rng.permutation(len(all_ids))
        cut = int(0.8 * len(all_ids))
        train_pool = [all_ids[k] for k in perm[:cut]]
        test_ids   = [all_ids[k] for k in perm[cut:]]

    # Carve a validation slice out of the training pool (deterministic).
    rng = np.random.RandomState(cfg.SEED)
    perm = rng.permutation(len(train_pool))
    n_val = max(1, int(val_frac * len(train_pool)))
    val_ids   = [train_pool[k] for k in perm[:n_val]]
    train_ids = [train_pool[k] for k in perm[n_val:]]
    return train_ids, val_ids, test_ids


train_ids, val_ids, test_ids = build_idrid_splits(cfg)
print(f'Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}')

train_ds = IDRiDSegmentationDataset(train_ids, cfg, train_tf, cfg.USE_CLAHE)
val_ds   = IDRiDSegmentationDataset(val_ids,   cfg, eval_tf,  cfg.USE_CLAHE)
test_ds  = IDRiDSegmentationDataset(test_ids,  cfg, eval_tf,  cfg.USE_CLAHE)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)


In [ ]:
# Sanity check: overlay the four lesion channels on one training image.
def denormalize(img_tensor):
    '''Undo ImageNet normalisation for display.'''
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

_img, _mask, _id = train_ds[0]
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
axes[0].imshow(denormalize(_img)); axes[0].set_title(f'{_id} (CLAHE)'); axes[0].axis('off')
for k, name in enumerate(cfg.LESIONS.keys()):
    axes[k + 1].imshow(_mask[k].numpy(), cmap='gray')
    axes[k + 1].set_title(name); axes[k + 1].axis('off')
plt.tight_layout(); plt.show()


## 4. Model - U-Net++ with a pretrained EfficientNet encoder

We use `segmentation-models-pytorch`, the production-standard library for encoder-decoder
segmentation. The choice is deliberate:

- **U-Net++** adds nested, densely connected skip pathways over plain U-Net. Its
  intermediate supervision recovers fine detail, which is exactly what tiny lesions need.
- **EfficientNet-B4 encoder, ImageNet-pretrained.** With only ~46 training images,
  transfer learning is essential; EfficientNet gives a strong accuracy/compute trade-off
  that fits a T4.
- **4-channel sigmoid head** for multi-label lesion prediction.

`deeplabv3plus` is available via the same config switch if you want to compare its ASPP
multi-scale context against U-Net++.


In [ ]:
def build_model(cfg):
    '''Construct the segmentation model from the config switches.'''
    common = dict(
        encoder_name=cfg.ENCODER,
        encoder_weights=cfg.ENCODER_WEIGHTS,
        in_channels=3,
        classes=cfg.NUM_LESIONS,   # one channel per lesion type
        activation=None,           # return raw logits; we apply sigmoid in loss/metrics
    )
    if cfg.ARCH == 'unetplusplus':
        model = smp.UnetPlusPlus(**common)
    elif cfg.ARCH == 'deeplabv3plus':
        model = smp.DeepLabV3Plus(**common)
    else:
        raise ValueError(f'Unknown ARCH: {cfg.ARCH}')
    return model

model = build_model(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'{cfg.ARCH} / {cfg.ENCODER}: {n_params:.1f}M parameters')


## 5. Loss and metrics

**Loss = Dice + Focal-Tversky.** Retinal lesions occupy well under 1% of the pixels, so a
plain cross-entropy is dominated by background. Dice loss optimises overlap directly.
Focal-Tversky loss generalises Dice with two knobs: `alpha` upweights false negatives
(critical - missing a microaneurysm is worse than a false alarm), and `gamma < 1` focuses
the gradient on the hard, rare lesion pixels. The literature on skin- and retinal-lesion
segmentation reports consistent gains from this family over Dice or BCE alone.

**Metrics.** We report per-lesion and mean IoU and Dice at a fixed threshold. Crucially,
we *aggregate the confusion counts (TP/FP/FN) across the whole set per lesion and then
compute the score once* (dataset-level micro), rather than averaging per-image scores.
This matters on IDRiD: most images do not contain every lesion type, and per-image
averaging with the usual zero-division convention silently inflates the numbers for the
absent lesions. Dataset-level aggregation is also how the IDRiD leaderboard reports.


In [ ]:
class FocalTverskyLoss(nn.Module):
    '''Focal-Tversky loss for multi-label segmentation.

    Tversky index = TP / (TP + alpha*FN + beta*FP). With alpha>beta we penalise missed
    lesion pixels more than false positives. Raising the result to gamma (<1) makes the
    loss concentrate on hard examples where the index is low.
    '''
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1.0):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        # Flatten spatial dims, keep batch*channel so each lesion map contributes.
        probs = probs.reshape(probs.shape[0], probs.shape[1], -1)
        targets = targets.reshape(targets.shape[0], targets.shape[1], -1)
        tp = (probs * targets).sum(-1)
        fn = ((1 - probs) * targets).sum(-1)
        fp = (probs * (1 - targets)).sum(-1)
        tversky = (tp + self.smooth) / (tp + self.alpha * fn + self.beta * fp + self.smooth)
        return torch.pow(1 - tversky, self.gamma).mean()


dice_loss = smp.losses.DiceLoss(mode='multilabel', from_logits=True)
tversky_loss = FocalTverskyLoss(cfg.TVERSKY_ALPHA, cfg.TVERSKY_BETA, cfg.TVERSKY_GAMMA)

def criterion(logits, targets):
    '''Weighted sum of Dice and Focal-Tversky.'''
    return cfg.LOSS_DICE_W * dice_loss(logits, targets) + \
           cfg.LOSS_TVERSKY_W * tversky_loss(logits, targets)


@torch.no_grad()
def batch_stats(logits, targets, threshold=0.5):
    '''Return per-lesion TP/FP/FN summed over the batch (shape [NUM_LESIONS] each).

    We deliberately return raw counts, not scores, so the caller can accumulate them
    across the whole loader and compute a single dataset-level score per lesion.
    '''
    preds = (torch.sigmoid(logits) > threshold).long()
    tp, fp, fn, tn = smp.metrics.get_stats(preds, targets.long(), mode='multilabel')
    return tp.sum(0), fp.sum(0), fn.sum(0)   # sum over the batch dimension -> [C]


def scores_from_counts(tp, fp, fn, eps=1e-7):
    '''IoU and Dice from aggregated counts. Returns numpy arrays.'''
    tp = tp.float(); fp = fp.float(); fn = fn.float()
    iou  = (tp / (tp + fp + fn + eps)).cpu().numpy()
    dice = (2 * tp / (2 * tp + fp + fn + eps)).cpu().numpy()
    return iou, dice


## 6. Checkpointing and resume (survives Colab disconnects)

Colab VMs are preempted without warning. We therefore write two checkpoints to Drive:

- `last.pt` - written every epoch, holds the full training state (model, optimiser,
  scheduler, AMP scaler, epoch counter, history and all RNG states).
- `best.pt` - written only when validation mean-Dice improves.

On start-up the loop looks for `last.pt` and resumes exactly where it stopped, so a
disconnected session can be continued by simply re-running the training cell.


In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, scaler,
                    epoch, best_dice, history, patience_counter):
    '''Persist the complete training state, including RNG, for exact resume.'''
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_dice': best_dice,
        'history': history,
        'patience_counter': patience_counter,
        'rng': {
            'python': random.getstate(),
            'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all(),
        },
        'cfg': {k: getattr(cfg, k) for k in dir(cfg) if k.isupper()},
    }, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
    '''Restore state saved by save_checkpoint. Returns the resume bookkeeping.'''
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    if optimizer is not None: optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler is not None: scheduler.load_state_dict(ckpt['scheduler'])
    if scaler is not None:    scaler.load_state_dict(ckpt['scaler'])
    if 'rng' in ckpt:
        random.setstate(ckpt['rng']['python'])
        np.random.set_state(ckpt['rng']['numpy'])
        torch.set_rng_state(ckpt['rng']['torch'])
        torch.cuda.set_rng_state_all(ckpt['rng']['cuda'])
    return ckpt

LAST_CKPT = f'{cfg.CKPT_DIR}/last.pt'
BEST_CKPT = f'{cfg.CKPT_DIR}/best.pt'


## 7. Train / validate loop

Mixed precision (AMP) roughly halves memory and speeds up the T4. Gradient accumulation
lets us keep an effective batch of 8 while only 4 images sit in memory at once. A
per-batch `try/except` skips a corrupt sample instead of killing a multi-hour run.


In [ ]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader, optimizer, scheduler, scaler, cfg):
    model.train()
    running = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, (images, masks, _) in enumerate(tqdm(loader, desc='train', leave=False)):
        try:
            images, masks = images.to(device), masks.to(device)
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(images)
                loss = criterion(logits, masks) / cfg.GRAD_ACCUM
            scaler.scale(loss).backward()
            # Step the optimiser every GRAD_ACCUM micro-batches.
            if (step + 1) % cfg.GRAD_ACCUM == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()             # OneCycle steps per optimiser update
            running += loss.item() * cfg.GRAD_ACCUM * images.size(0)
        except RuntimeError as e:
            # Typically an OOM on a pathological batch - free memory and continue.
            print('  [warn] skipped a batch:', str(e)[:120])
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    return running / len(loader.dataset)


@torch.no_grad()
def validate(model, loader, cfg):
    model.eval()
    running = 0.0
    # Accumulate confusion counts across the whole validation set, per lesion.
    tp_t = torch.zeros(cfg.NUM_LESIONS, device=device)
    fp_t = torch.zeros(cfg.NUM_LESIONS, device=device)
    fn_t = torch.zeros(cfg.NUM_LESIONS, device=device)
    for images, masks, _ in tqdm(loader, desc='val', leave=False):
        images, masks = images.to(device), masks.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(images)
            loss = criterion(logits, masks)
        running += loss.item() * images.size(0)
        tp, fp, fn = batch_stats(logits.float(), masks, cfg.THRESHOLD)
        tp_t += tp; fp_t += fp; fn_t += fn
    iou, dice = scores_from_counts(tp_t, fp_t, fn_t)
    return running / len(loader.dataset), iou, dice


In [ ]:
# Optimiser / scheduler / AMP scaler. OneCycle warms LR up then anneals it, which suits
# short fine-tuning runs on small datasets.
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
steps_per_epoch = max(1, len(train_loader) // cfg.GRAD_ACCUM)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=cfg.LR, epochs=cfg.NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch, pct_start=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

# Resume bookkeeping.
start_epoch = 0
best_dice = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': []}

# Auto-resume if a previous session left a checkpoint.
if os.path.exists(LAST_CKPT):
    print('Resuming from', LAST_CKPT)
    ckpt = load_checkpoint(LAST_CKPT, model, optimizer, scheduler, scaler)
    start_epoch = ckpt['epoch'] + 1
    best_dice = ckpt['best_dice']
    patience_counter = ckpt['patience_counter']
    history = ckpt['history']
    print(f'  -> continuing at epoch {start_epoch}, best Dice so far {best_dice:.4f}')


In [ ]:
# The training loop itself. Re-running this cell after a disconnect continues cleanly
# because all state was restored above.
for epoch in range(start_epoch, cfg.NUM_EPOCHS):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, cfg)
    val_loss, val_iou, val_dice = validate(model, val_loader, cfg)
    mean_dice = float(np.mean(val_dice))
    mean_iou = float(np.mean(val_iou))

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(mean_iou)
    history['val_dice'].append(mean_dice)

    print(f"Epoch {epoch+1:02d}/{cfg.NUM_EPOCHS} "
          f"| train {train_loss:.4f} | val {val_loss:.4f} "
          f"| mDice {mean_dice:.4f} | mIoU {mean_iou:.4f} "
          f"| {time.time()-t0:.0f}s")
    print('    per-lesion Dice:',
          {n: round(float(d), 3) for n, d in zip(cfg.LESIONS, val_dice)})

    # Always refresh last.pt so a disconnect loses at most the current epoch.
    save_checkpoint(LAST_CKPT, model, optimizer, scheduler, scaler,
                    epoch, best_dice, history, patience_counter)

    if mean_dice > best_dice:
        best_dice = mean_dice
        patience_counter = 0
        save_checkpoint(BEST_CKPT, model, optimizer, scheduler, scaler,
                        epoch, best_dice, history, patience_counter)
        print(f'    -> new best mean Dice {best_dice:.4f} (saved best.pt)')
    else:
        patience_counter += 1
        if patience_counter >= cfg.PATIENCE:
            print(f'Early stopping: no improvement for {cfg.PATIENCE} epochs.')
            break

print(f'\nTraining finished. Best validation mean Dice: {best_dice:.4f}')


In [ ]:
# Training curves.
if len(history['train_loss']) > 0:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(history['train_loss'], label='train loss')
    ax[0].plot(history['val_loss'], label='val loss')
    ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
    ax[1].plot(history['val_dice'], label='val mean Dice')
    ax[1].plot(history['val_iou'], label='val mean IoU')
    ax[1].set_title('Validation overlap'); ax[1].set_xlabel('epoch'); ax[1].legend()
    plt.tight_layout()
    plt.savefig(f'{cfg.OUTPUT_DIR}/seg_training_curves.png', dpi=120)
    plt.show()


## 8. Evaluation on the IDRiD test split

We reload the best checkpoint and report IoU/Dice per lesion and combined (the union of
all four channels - this "combined" mask is the exact target the Grad-CAM benchmark also
uses, so the two tracks are directly comparable). We also sweep the threshold to report
the best operating point, since the optimal cut differs between abundant exudates and
sparse microaneurysms.


In [ ]:
# Load the best model for evaluation.
if os.path.exists(BEST_CKPT):
    load_checkpoint(BEST_CKPT, model)
    print('Loaded best.pt for evaluation.')
model.eval()


@torch.no_grad()
def collect_predictions(model, loader):
    '''Return probabilities and ground truth for the whole loader (small test set).'''
    all_probs, all_masks, ids = [], [], []
    for images, masks, image_id in loader:
        images = images.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            probs = torch.sigmoid(model(images)).float().cpu()
        all_probs.append(probs)
        all_masks.append(masks)
        ids.extend(image_id)
    return torch.cat(all_probs), torch.cat(all_masks), ids

probs_test, masks_test, ids_test = collect_predictions(model, test_loader)
print('Test predictions:', tuple(probs_test.shape))


In [ ]:
def score_at_threshold(probs, masks, threshold):
    '''Per-lesion and combined IoU/Dice at a given threshold (dataset-level aggregation).'''
    preds = (probs > threshold).long()
    tp, fp, fn, tn = smp.metrics.get_stats(preds, masks.long(), mode='multilabel')
    # Sum counts over all images per lesion, then score once (avoids empty-image inflation).
    iou, dice = scores_from_counts(tp.sum(0), fp.sum(0), fn.sum(0))

    # Combined = union across the 4 lesion channels, scored as a single binary map.
    comb_pred = preds.max(dim=1, keepdim=True).values
    comb_gt   = masks.long().max(dim=1, keepdim=True).values
    ctp, cfp, cfn, ctn = smp.metrics.get_stats(comb_pred, comb_gt, mode='multilabel')
    comb_iou, comb_dice = scores_from_counts(ctp.sum(0), cfp.sum(0), cfn.sum(0))
    return iou, dice, float(comb_iou[0]), float(comb_dice[0])


# Threshold sweep on mean per-lesion Dice.
sweep = {}
for t in np.round(np.arange(0.1, 0.9, 0.05), 2):
    _, dice, _, _ = score_at_threshold(probs_test, masks_test, t)
    sweep[t] = float(np.mean(dice))
best_t = max(sweep, key=sweep.get)
print('Best threshold by mean Dice:', best_t)

iou, dice, comb_iou, comb_dice = score_at_threshold(probs_test, masks_test, best_t)
seg_table = pd.DataFrame({
    'Lesion': list(cfg.LESIONS.keys()) + ['Combined'],
    'IoU':  list(np.round(iou, 4)) + [round(comb_iou, 4)],
    'Dice': list(np.round(dice, 4)) + [round(comb_dice, 4)],
})
seg_table.to_csv(f'{cfg.OUTPUT_DIR}/segmentation_test_results.csv', index=False)
print(seg_table.to_string(index=False))


In [ ]:
# Qualitative panel: image | combined GT | combined prediction, for a few test images.
n_show = min(4, len(ids_test))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
if n_show == 1:
    axes = axes[None, :]
for r in range(n_show):
    img_t, mask_t, image_id = test_ds[r]
    gt_comb = mask_t.max(0).values.numpy()
    pred_comb = (probs_test[r].max(0).values > best_t).numpy().astype(float)
    axes[r, 0].imshow(denormalize(img_t)); axes[r, 0].set_title(f'{image_id}'); axes[r, 0].axis('off')
    axes[r, 1].imshow(gt_comb, cmap='gray'); axes[r, 1].set_title('GT (all lesions)'); axes[r, 1].axis('off')
    axes[r, 2].imshow(pred_comb, cmap='gray'); axes[r, 2].set_title('U-Net++ prediction'); axes[r, 2].axis('off')
plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/segmentation_qualitative.png', dpi=120)
plt.show()


## 9. Classical baseline (reference point)

Before deep learning, exudates were segmented with morphology and thresholding. We
include one compact classical pipeline for **hard exudates** - CLAHE, morphological
top-hat to isolate small bright structures, then Otsu thresholding - and score it with
the same IoU/Dice. It sets an honest floor that the learned model should clear, and
directly answers the question "how much does the deep model actually add over classical
image processing?".


In [ ]:
def classical_exudate_segment(image_rgb):
    '''Morphology + Otsu baseline for bright lesions (hard exudates).'''
    green = image_rgb[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)
    # White top-hat keeps bright structures smaller than the structuring element.
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    tophat = cv2.morphologyEx(enhanced, cv2.MORPH_TOPHAT, kernel)
    _, binary = cv2.threshold(tophat, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary.astype(np.uint8)


# Evaluate the classical baseline on the hard-exudate channel over the test set.
ex_index = list(cfg.LESIONS.keys()).index('HardExudates')
cl_iou, cl_dice, n_used = 0.0, 0.0, 0
for image_id in ids_test:
    image, crop_box = load_fundus(f'{cfg.IDRID_IMG_DIR}/{image_id}.jpg', use_clahe=False)
    image = cv2.resize(image, (cfg.IMG_SIZE, cfg.IMG_SIZE))
    pred = classical_exudate_segment(image)
    # Ground-truth EX mask, cropped + resized the same way.
    gt_path = f'{cfg.IDRID_MASK_DIR}/HardExudates/{image_id}_EX.tif'
    if not os.path.exists(gt_path):
        continue
    r0, r1, c0, c1 = crop_box
    gt = np.array(Image.open(gt_path).convert('L'))[r0:r1, c0:c1]
    gt = cv2.resize(gt, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST) > 0
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    cl_iou += inter / union if union > 0 else 0.0
    cl_dice += 2 * inter / (pred.sum() + gt.sum()) if (pred.sum() + gt.sum()) > 0 else 0.0
    n_used += 1
if n_used:
    print(f'Classical hard-exudate baseline over {n_used} images: '
          f'IoU {cl_iou/n_used:.4f} | Dice {cl_dice/n_used:.4f}')
    print(f'U-Net++ hard-exudate (same channel):     '
          f'IoU {iou[ex_index]:.4f} | Dice {dice[ex_index]:.4f}')


# Track B - Explainability benchmark

This track preserves the original project identity: probe a **classifier** with the
Grad-CAM family and score the heatmaps against the IDRiD masks. It is optional - if you
only care about IoU/Dice, Track A above is the answer. Its value here is scientific: it
quantifies the gap between weakly-supervised saliency and the supervised segmenter.

We fix two things versus the original run:

1. Add **Layer-CAM** and **HiRes-CAM**, which produce sharper, higher-resolution maps
   than vanilla Grad-CAM at a mid-level layer.
2. Keep **Eigen-CAM** only as a documented negative control. Eigen-CAM ignores the class
   target, so it cannot be expected to localise a specific lesion set - its low score is
   the point, not a failure to tune.


## 10. Classifier for the CAM track

The severity classifier is trained on the APTOS-2019 subset. If you already trained one
(`best_model.pth` from the earlier version), it is loaded; otherwise a short training run
produces one. This is intentionally lightweight - the classifier is a probe, not the
deliverable.


In [ ]:
from torchvision import models, transforms as T

APTOS_CKPT = f'{cfg.OUTPUT_DIR}/aptos_classifier.pth'
NUM_CLASSES = 5

def build_classifier():
    m = models.efficientnet_b0(weights='IMAGENET1K_V1')
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    return m

clf = build_classifier().to(device)

clf_tf = T.Compose([
    T.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [ ]:
class APTOSDataset(Dataset):
    '''Minimal APTOS loader for the classifier probe.'''
    def __init__(self, csv_file, img_dir, transform):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(f"{self.img_dir}/{row['id_code']}.png").convert('RGB')
        return self.transform(img), int(row['diagnosis'])


TRAIN_CLASSIFIER = not os.path.exists(APTOS_CKPT)   # set True to force retraining

if os.path.exists(APTOS_CKPT):
    clf.load_state_dict(torch.load(APTOS_CKPT, map_location=device))
    print('Loaded existing APTOS classifier.')
elif os.path.exists(cfg.APTOS_CSV):
    print('Training a short APTOS classifier for the CAM probe...')
    aptos_ds = APTOSDataset(cfg.APTOS_CSV, cfg.APTOS_IMG_DIR, clf_tf)
    aptos_loader = DataLoader(aptos_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                              num_workers=cfg.NUM_WORKERS, pin_memory=True)
    # Class-weighted loss to offset the APTOS "No DR" majority.
    counts = np.bincount(pd.read_csv(cfg.APTOS_CSV)['diagnosis'].values, minlength=NUM_CLASSES)
    weights = counts.sum() / (NUM_CLASSES * np.maximum(counts, 1))
    clf_criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32).to(device), label_smoothing=0.1)
    clf_opt = torch.optim.AdamW(clf.parameters(), lr=5e-5, weight_decay=1e-4)
    clf_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    clf.train()
    for ep in range(15):
        tot = 0.0
        for x, y in tqdm(aptos_loader, desc=f'clf epoch {ep+1}', leave=False):
            x, y = x.to(device), y.to(device)
            clf_opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                loss = clf_criterion(clf(x), y)
            clf_scaler.scale(loss).backward()
            clf_scaler.step(clf_opt); clf_scaler.update()
            tot += loss.item() * x.size(0)
        print(f'  clf epoch {ep+1}: loss {tot/len(aptos_ds):.4f}')
    torch.save(clf.state_dict(), APTOS_CKPT)
    print('Saved APTOS classifier ->', APTOS_CKPT)
else:
    print('No APTOS data found; Track B will be skipped. Track A is unaffected.')

clf.eval()


In [ ]:
# Inspect the spatial resolution of each EfficientNet block so we can pick a mid-level
# layer (~14x14) that is finer than the final 7x7 map the original notebook used.
dummy = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE).to(device)
shapes = {}
def _hook(name):
    def fn(_m, _i, o): shapes[name] = tuple(o.shape)
    return fn
handles = [blk.register_forward_hook(_hook(f'features[{i}]'))
           for i, blk in enumerate(clf.features)]
with torch.no_grad():
    _ = clf(dummy)
for h in handles: h.remove()
for k, v in shapes.items():
    print(k, v)


In [ ]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM, LayerCAM, HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Mid-level target layer for finer localisation than the default last block.
FINE_LAYER_INDEX = 5
target_layers = [clf.features[FINE_LAYER_INDEX]]
print('CAM target layer:', f'features[{FINE_LAYER_INDEX}]', shapes.get(f'features[{FINE_LAYER_INDEX}]'))

cam_methods = {
    'GradCAM':    GradCAM(model=clf, target_layers=target_layers),
    'GradCAM++':  GradCAMPlusPlus(model=clf, target_layers=target_layers),
    'LayerCAM':   LayerCAM(model=clf, target_layers=target_layers),
    'HiResCAM':   HiResCAM(model=clf, target_layers=target_layers),
    'EigenCAM':   EigenCAM(model=clf, target_layers=target_layers),   # class-agnostic control
}
print('CAM methods:', list(cam_methods.keys()))


## 11. Score the CAM heatmaps against IDRiD masks

For each test image we take the model's predicted severity as the CAM target, threshold
the heatmap with Otsu (adaptive per image), and compute IoU/Dice against the combined
lesion mask - the same combined target Track A reports, so the numbers line up in one
table.


In [ ]:
from skimage.filters import threshold_otsu
from scipy.ndimage import gaussian_filter

def combined_mask_for(image_id, out_hw):
    '''Union of the four lesion masks, cropped + resized to out_hw (for the CAM track).'''
    image, crop_box = load_fundus(f'{cfg.IDRID_IMG_DIR}/{image_id}.jpg', use_clahe=False)
    r0, r1, c0, c1 = crop_box
    combined = np.zeros(out_hw, dtype=bool)
    for name, suf in cfg.LESIONS.items():
        p = f'{cfg.IDRID_MASK_DIR}/{name}/{image_id}_{suf}.tif'
        if os.path.exists(p):
            m = np.array(Image.open(p).convert('L'))[r0:r1, c0:c1]
            m = cv2.resize(m, out_hw[::-1], interpolation=cv2.INTER_NEAREST) > 0
            combined |= m
    return combined.astype(np.uint8)


def binarize_heatmap(heatmap, blur_sigma=1.0):
    '''Otsu threshold with a small blur; safe fallback for flat maps.'''
    sm = gaussian_filter(heatmap, sigma=blur_sigma) if blur_sigma > 0 else heatmap
    try:
        thr = threshold_otsu(sm)
    except ValueError:
        thr = 0.5
    return (sm >= thr).astype(np.uint8)


def iou_dice(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    iou = inter / union if union > 0 else 0.0
    dice = 2 * inter / (pred.sum() + gt.sum()) if (pred.sum() + gt.sum()) > 0 else 0.0
    return iou, dice


In [ ]:
# Only run the CAM benchmark if the classifier is available.
run_cam = os.path.exists(APTOS_CKPT) or ('aptos_ds' in dir())
cam_rows = []
if run_cam:
    for image_id in tqdm(ids_test, desc='CAM benchmark'):
        image = Image.open(f'{cfg.IDRID_IMG_DIR}/{image_id}.jpg').convert('RGB')
        x = clf_tf(image).unsqueeze(0).to(device)
        with torch.no_grad():
            pred_class = int(clf(x).argmax(1).item())
        targets = [ClassifierOutputTarget(pred_class)]
        gt = combined_mask_for(image_id, (cfg.IMG_SIZE, cfg.IMG_SIZE))
        row = {'image_id': image_id, 'pred_class': pred_class}
        for name, cam in cam_methods.items():
            hm = cam(input_tensor=x, targets=targets)[0]
            pred = binarize_heatmap(hm)
            i, d = iou_dice(pred, gt)
            row[f'{name}_IoU'] = i
            row[f'{name}_Dice'] = d
        cam_rows.append(row)
    cam_df = pd.DataFrame(cam_rows)
    cam_df.to_csv(f'{cfg.OUTPUT_DIR}/cam_benchmark_results.csv', index=False)
    print(cam_df.filter(like='Dice').mean().round(4))
else:
    cam_df = pd.DataFrame()
    print('CAM track skipped (no classifier).')


## 12. Head-to-head: supervised segmentation vs. weakly-supervised CAM

The final chart puts the U-Net++ combined IoU/Dice next to every CAM method on the same
axis. This is the benchmark the repository is named for - now with a supervised upper
bound to make the comparison meaningful.


In [ ]:
labels, iou_vals, dice_vals = [], [], []

# Supervised segmentation (combined) first as the reference bar.
labels.append('U-Net++ (supervised)')
iou_vals.append(comb_iou)
dice_vals.append(comb_dice)

if not cam_df.empty:
    for name in cam_methods.keys():
        labels.append(name)
        iou_vals.append(float(cam_df[f'{name}_IoU'].mean()))
        dice_vals.append(float(cam_df[f'{name}_Dice'].mean()))

x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, iou_vals, w, label='IoU')
ax.bar(x + w/2, dice_vals, w, label='Dice')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_ylabel('Score (combined lesion mask)')
ax.set_title('Lesion localisation: supervised segmentation vs. CAM explainability')
ax.legend()
for i, (iv, dv) in enumerate(zip(iou_vals, dice_vals)):
    ax.text(i - w/2, iv + 0.005, f'{iv:.2f}', ha='center', va='bottom', fontsize=8)
    ax.text(i + w/2, dv + 0.005, f'{dv:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/benchmark_comparison.png', dpi=120)
plt.show()


## 13. Summary

- **Track A (U-Net++ supervised segmentation)** is the recommended model for lesion
  localisation and is what raises IoU/Dice into the range the IDRiD literature reports.
- **Track B (CAM benchmark)** stays as an interpretability study and now shows, on a
  shared axis, how far saliency maps sit below a supervised segmenter - with Eigen-CAM
  behaving as the expected class-agnostic lower bound.

All checkpoints are on Drive under `checkpoints/` (resume-safe), and every table and
figure is written to `outputs/`. Re-running the training cell after a disconnect resumes
automatically from `last.pt`.
